# AgenticEd Labs · 00 — Environment Check

<a href="https://colab.research.google.com/github/AgenticEdOrg/AgenticEd/blob/main/labs/00_environment_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run this once before Week 1. It confirms your Python version, the pinned packages, your API key, a real model call, the course MCP server, and the course data. Every check prints **PASS** or **FAIL** with a fix.

**You need one API key** (bring your own): an OpenAI key (`OPENAI_API_KEY`) or an Anthropic key (`ANTHROPIC_API_KEY`).

- **Colab:** click the key icon in the left sidebar → *Add new secret* → name it exactly as above → paste the key → switch on *Notebook access*.
- **Local:** `cd labs && python -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt`, then copy `.env.example` to `.env` and paste your key.

The labs default to low-cost models (`gpt-5.4-mini` or `claude-haiku-5-5`). Running every notebook end to end typically costs well under US$1. Set a monthly spend limit in your provider's dashboard anyway — it's a good production habit.

In [ ]:
# ── Setup (run once) ─────────────────────────────────────────────────────────
# In Colab: clones the course repo and installs the pinned requirements (2–3 minutes the first time).
# If Colab asks you to restart the runtime after installing, do so and run this cell again.
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/AgenticEd").exists():
        !git clone -q --depth 1 https://github.com/AgenticEdOrg/AgenticEd.git /content/AgenticEd
    %cd /content/AgenticEd/labs
    %pip install -q -r requirements.txt
sys.path.insert(0, str(pathlib.Path.cwd()))

import labkit
cfg = labkit.setup()          # reads your key, picks the model, prints what it chose

## 1 · Python and packages

In [ ]:
import importlib.metadata as md, platform
checks = []
def record(name, ok, detail=""):
    checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})
    print(f"{'PASS' if ok else 'FAIL'}  {name}  {detail}")

py = tuple(int(x) for x in platform.python_version_tuple()[:2])
record("Python >= 3.11", py >= (3, 11), platform.python_version())

for pkg in ["openai", "anthropic", "litellm", "pydantic-ai", "pydantic-evals", "openai-agents", "langgraph",
            "langchain-openai", "langchain-anthropic", "crewai", "chromadb", "mcp", "sentence-transformers",
            "rank-bm25", "tiktoken", "presidio-analyzer"]:
    try:
        record(f"package {pkg}", True, md.version(pkg))
    except md.PackageNotFoundError:
        optional = pkg == "crewai"
        record(f"package {pkg}", optional, "not installed" + (" (optional — only Week 4's CrewAI section)" if optional else " → pip install -r requirements.txt"))

## 2 · API key and a real model call

In [ ]:
import os, time
key_name = "OPENAI_API_KEY" if cfg.provider == "openai" else "ANTHROPIC_API_KEY"
key = os.environ.get(key_name, "")
record(f"{key_name} present", bool(key), (key[:6] + "…" + key[-4:]) if len(key) > 12 else "(set)")

from pydantic_ai import Agent
t = time.time()
try:
    result = await Agent(cfg.pydantic_ai).run("Reply with exactly: environment ok")
    u = result.usage
    record("model call", "ok" in result.output.lower(),
           f"{cfg} · {time.time()-t:.1f}s · {u.input_tokens} in / {u.output_tokens} out tokens · ~${cfg.cost(u.input_tokens, u.output_tokens) or 'n/a'}")
except Exception as e:
    record("model call", False, f"{type(e).__name__}: {str(e)[:200]}")

## 3 · Tool calling (every lab depends on it)

In [ ]:
tool_agent = Agent(cfg.pydantic_ai, instructions="Always use the multiply tool for multiplication.")

@tool_agent.tool_plain
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

try:
    r = await tool_agent.run("What is 1234 times 5678?")
    used = any(getattr(p, "tool_name", None) == "multiply" for m in r.all_messages() for p in getattr(m, "parts", []))
    record("native tool calling", used and "7006652" in r.output.replace(",", ""), r.output[:80])
except Exception as e:
    record("native tool calling", False, f"{type(e).__name__}: {str(e)[:200]}")

## 4 · Course MCP server and data

In [ ]:
from pydantic_ai.mcp import MCPToolset
from pydantic_ai.models.test import TestModel   # a fake model that calls every tool once — no API cost

try:
    probe = Agent(TestModel(), toolsets=[MCPToolset("mcp_servers/policy_server.py")])
    r = await probe.run("probe")
    names = sorted({p.tool_name for m in r.all_messages() for p in getattr(m, "parts", []) if hasattr(p, "tool_name")})
    record("MCP policy server", {"get_policy", "list_policies", "search_policies"} <= set(names), ", ".join(names))
except Exception as e:
    record("MCP policy server", False, f"{type(e).__name__}: {str(e)[:200]}")

docs = labkit.load_corpus()
record("course corpus", len(docs) >= 7, f"{len(docs)} documents")
record("RAG eval set", len(labkit.load_jsonl("eval/rag_questions.jsonl")) == 20, "20 questions")

## Summary

In [ ]:
import pandas as pd
summary = pd.DataFrame(checks)
display(summary.style.map(lambda v: "color: #047857; font-weight: 600" if v == "PASS" else
                          ("color: #B91C1C; font-weight: 600" if v == "FAIL" else ""), subset=["result"]))
failed = summary[summary.result == "FAIL"]
print("\nAll checks passed — you're ready for Week 1." if failed.empty else f"\n{len(failed)} check(s) failed — fix them before Week 1.")

### Troubleshooting

| Symptom | Fix |
|---|---|
| `No API key found` | Add the secret with the exact name, enable *Notebook access*, then re-run the setup cell. |
| `401` / `invalid_api_key` | The key was copied incompletely or revoked — create a new one. |
| `429` / rate limit | New accounts have low limits. Wait a minute, or add a small prepaid balance. |
| `model not found` | Your account may not have the default model. Set `AGENTICED_MODEL`, e.g. `openai:gpt-4.1-mini`. |
| Colab "restart runtime" prompt | Restart, then run the setup cell again. |
| Using Ollama / vLLM / a LiteLLM proxy | Set `OPENAI_BASE_URL` and `AGENTICED_MODEL=openai:<model>` (the model must support tool calling). |